<a href="https://colab.research.google.com/github/epi24/multimodal-meme-analysis/blob/main/roberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import RobertaModel, RobertaTokenizer
from tqdm.auto import tqdm
import gc
from google.colab import drive



PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
SAVE_DIR        = '/content/drive/MyDrive/only_text_roberta_NEW'
MODEL_NAME      = 'roberta_text_only_NEW'

CHECKPOINT_PATH = '/content/drive/MyDrive/only_text_roberta_NEW/roberta_text_only_NEW_epoch_12.pth'

# --- HYPERPARAMETER ---
BATCH_SIZE    = 256
EPOCHS        = 20
LEARNING_RATE = 2e-5    # 2e-5 ist der absolute Standard-Sweetspot für RoBERTa
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (NUR TEXT)
# ==========================================
class RobertaDataset(Dataset):
    def __init__(self, json_path, tokenizer):
        self.tokenizer = tokenizer
        self.samples = []
        self.label_map = {}
        self.id_to_label = {}

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Labels ermitteln und alphabetisch sortieren
        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        for idx, label in enumerate(unique_labels):
            self.label_map[label] = idx
            self.id_to_label[idx] = label

        # Map speichern
        with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
            json.dump(self.id_to_label, f)

        for item in tqdm(raw_data, desc="Lade Texte"):
            label_str = item.get('label')

            # OCR Text holen und bereinigen
            ocr_text = item.get('text', "").strip()
            # RoBERTa kann 512 Tokens aufnehmen, aber für Memes reichen oft weniger.
            if len(ocr_text) < 2: ocr_text = "empty meme text"

            self.samples.append({
                'text': ocr_text,
                'label': self.label_map[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} Texte geladen.\n")
        del raw_data
        gc.collect()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # RoBERTa Tokenizer aufrufen
        encoding = self.tokenizer(
            sample['text'],
            add_special_tokens=True,
            max_length=128,          # 128 reicht für Meme-Texte locker aus
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

# ==========================================
# 3. ARCHITEKTUR (RoBERTa)
# ==========================================
class RobertaMemeClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # RoBERTa Basis-Modell laden
        self.roberta = RobertaModel.from_pretrained('roberta-base')

        # --- PARTIAL UNFREEZING ---
        # Wir frieren die Basis ein
        for param in self.roberta.parameters():
            param.requires_grad = False

        # Wir tauen die letzte Transformer-Schicht (Layer 11) auf
        for param in self.roberta.encoder.layer[-1].parameters():
            param.requires_grad = True

        # Classifier Head (RoBERTa Output ist 768-dimensional)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        # Text durch RoBERTa jagen
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)

        # Wir nehmen den Vektor des <s> Tokens (Index 0), der den ganzen Satz repräsentiert
        sentence_representation = outputs.last_hidden_state[:, 0, :]

        # Klassifizieren
        return self.classifier(sentence_representation)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
def run_roberta_training():
    print("--- Start RoBERTa Text-Only Training ---")

    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

    train_dataset = RobertaDataset(PATH_TRAIN_JSON, tokenizer)
    val_dataset   = RobertaDataset(PATH_VAL_JSON, tokenizer)

    num_classes = len(train_dataset.label_map)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = RobertaMemeClassifier(num_classes).to(DEVICE)

    # Standard Optimizer mit einheitlicher Lernrate
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0
    best_acc = 0.0

    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"\n[INFO] Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = checkpoint["epoch"]
            best_acc = checkpoint.get("best_acc", 0.0)
            print(f"[SUCCESS] Geladen! Starte ab Epoche {start_epoch+1}.")
        else:
            model.load_state_dict(checkpoint)
            print("[WARNUNG] Alter Checkpoint (Nur Gewichte).")

    print(f"\nStarte Training bis Epoche {EPOCHS}...\n")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        del input_ids, attention_mask, labels, logits
        gc.collect()
        torch.cuda.empty_cache()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    logits = model(input_ids, attention_mask)

                _, preds = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                del input_ids, attention_mask, labels, logits

        val_acc = val_correct / val_total
        print(f" -> Resultat E{epoch+1}: Val Acc: {val_acc:.2%}")

        checkpoint_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc
        }

        torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{epoch+1}.pth"))

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_best.pth"))
            print(f"    [SAVED] Neues bestes Modell ({val_acc:.2%})!")

if __name__ == "__main__":
    run_roberta_training()

--- Start RoBERTa Text-Only Training ---


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Epoche 19 [Valid]:   0%|          | 0/37 [00:00<?, ?it/s]

 -> Resultat E19: Val Acc: 38.13%
    [SAVED] Neues bestes Modell (38.13%)!


Epoche 20/20 [Train]:   0%|          | 0/296 [00:00<?, ?it/s]

Epoche 20 [Valid]:   0%|          | 0/37 [00:00<?, ?it/s]

 -> Resultat E20: Val Acc: 38.60%
    [SAVED] Neues bestes Modell (38.60%)!


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizer
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
from google.colab import drive



# --- PFADE (NUR DAS TEST-SET!) ---

CHECKPOINT_PATH = '/content/drive/MyDrive/only_text_roberta_NEW/roberta_text_only_NEW_best.pth'
PATH_TEST_JSON  = '/content/drive/MyDrive/meme_test.json'
PATH_LABEL_MAP  = '/content/drive/MyDrive/only_text_roberta_NEW/roberta_text_only_NEW_map.json' # Die Map aus dem Training!
SAVE_DIR        = '/content/drive/MyDrive/only_text_roberta_NEW'
OUTPUT_CSV      = 'roberta_text_only_NEW_evaluation_results.csv'

BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (NUR TEST-DATEN)
# ==========================================
class EvalRobertaDataset(Dataset):
    def __init__(self, json_path, label_map_path, tokenizer):
        self.tokenizer = tokenizer
        self.samples = []

        # --- LADE DIE OFFIZIELLE LABEL MAP ---
        print(f"Lade offizielle Label-Map: {label_map_path.split('/')[-1]}")
        with open(label_map_path, 'r', encoding='utf-8') as f:
            loaded_map = json.load(f)
            self.id_to_label = {int(k): v for k, v in loaded_map.items()}
            self.label_map = {v: int(k) for k, v in loaded_map.items()}

        print(f"--- [EVAL DATASET] Lade Test-Texte aus {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        for item in tqdm(raw_data, desc="Lade Texte"):
            label_str = item.get('label')

            # Unbekannte Klassen im Testset ignorieren wir (Sicherheitscheck)
            if label_str not in self.label_map:
                continue

            filename = item.get('filename', 'unknown_file')
            ocr_text = item.get('text', "").strip()
            if len(ocr_text) < 2: ocr_text = "empty meme text"

            self.samples.append({
                'text': ocr_text,
                'label': self.label_map[label_str],
                'filename': filename
            })

        print(f"-> Bereit: {len(self.samples)} Test-Texte geladen.\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        encoding = self.tokenizer(
            sample['text'],
            add_special_tokens=True,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(sample['label'], dtype=torch.long),
            'filename': sample['filename'],
            'raw_text': sample['text']
        }

# ==========================================
# 3. ARCHITEKTUR (MUSS IDENTISCH SEIN)
# ==========================================
class RobertaMemeClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sentence_representation = outputs.last_hidden_state[:, 0, :]
        return self.classifier(sentence_representation)

# ==========================================
# 4. EVALUATION LOOP
# ==========================================
def run_roberta_evaluation():
    print("--- Starte RoBERTa Text-Only Test-Evaluation ---")

    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    eval_dataset = EvalRobertaDataset(PATH_TEST_JSON, PATH_LABEL_MAP, tokenizer)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    num_classes = len(eval_dataset.label_map)
    model = RobertaMemeClassifier(num_classes).to(DEVICE)

    # --- GEWICHTE LADEN ---
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
        print("[SUCCESS] Gewichte erfolgreich geladen.")
    else:
        print(f"[ERROR] Checkpoint nicht gefunden: {CHECKPOINT_PATH}")
        return

    model.eval()

    results_list = []
    all_preds = []
    all_labels = []

    print("Berechne Vorhersagen auf ungesehenen Daten...")

    with torch.no_grad():
        for batch in tqdm(eval_loader):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            filenames = batch['filename']
            raw_texts = batch['raw_text']

            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask)

            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(filenames)):
                pred_idx = preds[i].item()
                true_idx = labels[i].item()

                # Bereinige den Text für die CSV
                clean_text = raw_texts[i].replace("\n", " ").replace(";", ",")

                results_list.append({
                    "Dateiname": filenames[i],
                    "Wahre Klasse": eval_dataset.id_to_label[true_idx],
                    "Vorhersage": eval_dataset.id_to_label[pred_idx],
                    "Status": "KORREKT" if pred_idx == true_idx else "FALSCH",
                    "Sicherheit_Prozent": round(confidences[i].item() * 100, 2),
                    "OCR_Text": clean_text
                })

    # Gesamte Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n========================================")
    print(f"RoBERTa ACCURACY (Test Set): {acc:.2%}")
    print(f"========================================\n")

    # Metriken berechnen
    class_names = [eval_dataset.id_to_label[i] for i in range(num_classes)]
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

    # Detail-CSV speichern
    df_details = pd.DataFrame(results_list)
    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path_csv = os.path.join(SAVE_DIR, OUTPUT_CSV)
    df_details.to_csv(save_path_csv, index=False, sep=';', encoding='utf-8-sig')

    # Metriken-CSV speichern
    metrics_list = []
    for name in class_names:
        metrics = report_dict[name]
        metrics_list.append({
            "Meme": name,
            "Precision": round(metrics['precision'], 2),
            "Recall": round(metrics['recall'], 2),
            "F1-Score": round(metrics['f1-score'], 2),
            "Anzahl": metrics['support']
        })

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.sort_values(by="F1-Score", ascending=True)
    df_metrics.to_csv(os.path.join(SAVE_DIR, "roberta_text_only_NEW_metrics.csv"), index=False, sep=';')

    print("[FERTIG] Tabellen gespeichert. NLP-Baseline steht!")

if __name__ == "__main__":
    run_roberta_evaluation()


--- Starte RoBERTa Text-Only Test-Evaluation ---


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Lade Checkpoint: /content/drive/MyDrive/only_text_roberta_NEW/roberta_text_only_NEW_best.pth
[SUCCESS] Gewichte erfolgreich geladen.
Berechne Vorhersagen auf ungesehenen Daten...


  0%|          | 0/38 [00:00<?, ?it/s]


RoBERTa ACCURACY (Test Set): 37.99%

[FERTIG] Tabellen gespeichert. NLP-Baseline steht!


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
